In [1]:
#import libraries
import os
import time
import re
import json
import fitz
import tempfile
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from docx2pdf import convert

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Cấu hình ban đầu
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../Preprocessed_Data")

PAGE_THRESHOLD = 8
WORDS_PER_PAGE = 400

In [3]:
MODEL = "gemini-3.1-flash-lite"

In [4]:
# Prompt linh hoạt cho nhiều loại tài liệu quy chế
SYSTEM_INSTRUCTION = """
Bạn là chuyên gia tiền xử lý dữ liệu RAG cho hệ thống Quy chế đào tạo ĐHBK Hà Nội.

NHIỆM VỤ:
Phân tích tài liệu PDF hoặc .docx, sửa lỗi trình bày, chuyển bảng sang Markdown, và chia thành các chunk logic.
Giữ nguyên TOÀN BỘ nội dung tài liệu, bao gồm tất cả các điểm a), b), c)..., hệ số, mức phí, điều kiện cụ thể. Tuyệt đối không lược bỏ bất kỳ điều khoản hoặc chi tiết số liệu nào, dù nhỏ.

THÔNG TIN TÀI LIỆU:
- Tên file: {filename}
- parent_doc_id: {parent_doc_id}
- category: {category}
- year: {year}

CHIẾN LƯỢC CHIA CHUNK:
- Đơn vị cơ bản: mỗi Điều = 1 chunk
- Nếu Điều quá ngắn (< 80 từ): gộp với Điều liền kề cùng chủ đề
- Nếu Điều quá dài (> 600 từ): tách theo Khoản, mỗi Khoản là 1 chunk
- Bảng biểu lớn (> 5 hàng): có thể là chunk độc lập
- Phần mở đầu/định nghĩa chung: 1 chunk riêng
- Ranh giới Chương/Mục lớn (I, II, III... hoặc tương đương) LUÔN bắt đầu chunk mới, KHÔNG được gộp với nội dung của Chương/Mục trước dù đoạn đó < 80 từ

QUY TẮC ĐỊNH DẠNG:
1. Mỗi chunk nằm trong <<<CHUNK_START>>> ... <<<CHUNK_END>>>
2. Không trả về bất kỳ lời dẫn, giải thích hay hội thoại nào ngoài các thẻ trên
3. Bảng biểu dùng định dạng Markdown table
4. Hình ảnh/biểu đồ: mô tả bằng [Hình: <mô tả chi tiết tất cả những yếu tố quan trọng trong hình ảnh/biểu đồ có thể xuất hiện trong query>]
5. Trả lời hoàn toàn bằng tiếng Việt

CHIẾN LƯỢC METADATA CHO BẢNG BIỂU:
- topic_tags: liệt kê tất cả tên chương trình, học phần, đối tượng, mã số được đề cập trong bảng
  (ví dụ: nếu bảng có Global ICT, Logistics, Việt-Nhật thì tất cả phải có trong topic_tags)
- summary: phải đề cập ít nhất 2-3 đối tượng/giá trị cụ thể từ bảng, không chỉ mô tả chung chung
  Sai:  "Bảng học phí các chương trình đặc biệt ELITECH"
  Đúng: "Bảng học phí theo TCHP cho các chương trình ELITECH gồm Global ICT, Logistics,
         Việt-Nhật, với mức từ X đến Y triệu đồng/TCHP"

CẤU TRÚC CHUNK (tuân thủ chính xác, không thêm/bớt field):
<<<CHUNK_START>>>
source: {filename}
parent_doc_id: {parent_doc_id}
chunk_id: {doc_id}_001  ← tăng dần, 3 chữ số, ví dụ _002, _003
chunk_index: 1          ← số nguyên tăng dần
language: vi
category: {category}
year: {year}
chunk_title: [Tên điều khoản hoặc chủ đề chính]
topic_tags: [tag1, tag2, tag3]
summary: [2-3 câu tóm tắt, ưu tiên từ khóa: mã học phần, tín chỉ, cảnh báo học tập, điều kiện xét]

[Nội dung chunk ở đây]
<<<CHUNK_END>>>

VÍ DỤ OUTPUT:
<<<CHUNK_START>>>
source: Ngoai_ngu_2022_Quy_doi_CCTA.pdf
parent_doc_id: Ngoai_ngu_2022_Quy_doi_CCTA
chunk_id: Ngoai_ngu_2022_Quy_doi_CCTA_001
chunk_index: 1
language: vi
category: Ngoai_ngu
year: 2022
chunk_title: Điều 1 – Phạm vi áp dụng
topic_tags: [chứng chỉ ngoại ngữ, quy đổi, điều kiện tốt nghiệp]
summary: Quy định về việc quy đổi chứng chỉ ngoại ngữ quốc tế sang điểm học phần tương đương tại ĐHBK Hà Nội. Áp dụng cho sinh viên đại học chính quy có chứng chỉ IELTS, TOEIC, hoặc tương đương.

Điều 1. Phạm vi áp dụng

Quy định này áp dụng cho sinh viên đại học chính quy của Đại học Bách khoa Hà Nội có nhu cầu quy đổi chứng chỉ ngoại ngữ quốc tế...
<<<CHUNK_END>>>
"""

In [5]:
STRUCTURE_PROMPT = """
Phân tích cấu trúc tài liệu này và trả về JSON theo format sau. KHÔNG trả về gì khác ngoài JSON thuần:
[
  {"section": "Chương I - Quy định chung", "start_page": 1, "end_page": 4},
  {"section": "Chương II - ...", "start_page": 5, "end_page": 11}
]
Quy tắc:
- Đơn vị chia là Chương hoặc phần lớn tương đương (nếu không có Chương thì chia theo nhóm chủ đề)
- Mỗi phần tối thiểu 5 trang, tối đa 10 trang
- Ưu tiên tạo ÍT phần nhất có thể, gộp các Chương ngắn liền kề nếu tổng không vượt 10 trang
- Nếu 1 Chương > 10 trang: tách theo nhóm Điều, nhưng KHÔNG BAO GIỜ cắt giữa một Điều
- Không có markdown, không có code block, chỉ JSON
"""

In [6]:
#helper function to convert docx to pdf

def convert_docx_to_temp_pdf(docx_path: Path) -> Path:
    """Convert .docx sang PDF tạm, trả về path của PDF"""
    tmp_dir = Path(tempfile.mkdtemp())
    pdf_path = tmp_dir / docx_path.with_suffix(".pdf").name
    convert(docx_path, pdf_path)
    return pdf_path

In [7]:
#helper functions to deal with long pdf or word files

def get_page_count(file_path: Path) -> int:
    """Đếm trang PDF hoặc ước tính số trang cho docx"""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        doc = fitz.open(file_path)
        count = doc.page_count
        doc.close()
        return count
    elif suffix == ".docx":
        from docx import Document
        doc = Document(file_path)
        word_count = sum(len(p.text.split()) for p in doc.paragraphs)
        return max(1, word_count // WORDS_PER_PAGE)
    return 0


def extract_structure_from_pdf(file_path: Path) -> list[dict]:
    """Pass 1: Gemini phân tích cấu trúc PDF, trả về list các section với page range"""
    print("    [~] Đang phân tích cấu trúc tài liệu...")
    file_upload = client.files.upload(file=file_path)

    while file_upload.state.name == "PROCESSING":
        time.sleep(2)
        file_upload = client.files.get(name=file_upload.name)

    if file_upload.state.name == "FAILED":
        client.files.delete(name=file_upload.name)
        raise RuntimeError("Upload thất bại khi phân tích cấu trúc")

    response = client.models.generate_content(
        model=MODEL,
        contents=[file_upload, STRUCTURE_PROMPT],
        config=types.GenerateContentConfig(temperature=0.0)
    )
    client.files.delete(name=file_upload.name)

    # Xóa markdown code fence nếu Gemini vẫn thêm vào
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', response.text.strip())
    return json.loads(text)


def split_pdf(file_path: Path, structure: list[dict], output_folder: Path) -> list[Path]:
    """Cắt PDF theo cấu trúc từ Gemini"""
    doc = fitz.open(file_path)
    parts = []
    for i, section in enumerate(structure):
        start = section["start_page"] - 1   # fitz dùng 0-index
        end   = section["end_page"]   - 1
        sub   = fitz.open()
        sub.insert_pdf(doc, from_page=start, to_page=end)
        part_path = output_folder / f"{file_path.stem}_{i+1:02d}.pdf"
        sub.save(part_path)
        sub.close()
        parts.append(part_path)
    doc.close()
    return parts


def split_docx(file_path: Path, output_folder: Path) -> list[Path]:
    """Cắt docx theo ranh giới Chương, đảm bảo mỗi part ≤ PAGE_THRESHOLD trang ước tính"""
    import copy
    from docx import Document

    doc = Document(file_path)
    chapter_re = re.compile(r'^(Chương|CHƯƠNG)\s+[IVXLCDM\d]+', re.IGNORECASE)

    sections, current, current_words = [], [], 0
    for para in doc.paragraphs:
        is_boundary = chapter_re.match(para.text.strip())
        words = len(para.text.split())

        if is_boundary and current and current_words > WORDS_PER_PAGE * 2:
            sections.append(current)
            current, current_words = [], 0

        current.append(para)
        current_words += words

        if current_words >= WORDS_PER_PAGE * PAGE_THRESHOLD:
            sections.append(current)
            current, current_words = [], 0

    if current:
        sections.append(current)

    parts = []
    for i, paras in enumerate(sections):
        new_doc = Document()
        # Xóa paragraph mặc định rỗng
        for p in new_doc.paragraphs:
            p._element.getparent().remove(p._element)
        for para in paras:
            new_doc.add_paragraph()._p.getparent().replace(
                new_doc.paragraphs[-1]._p,
                copy.deepcopy(para._p)
            )
        part_path = output_folder / f"{file_path.stem}_{i+1:02d}.docx"
        new_doc.save(part_path)
        parts.append(part_path)
    return parts

In [8]:
def upload_and_process(file_path: Path, original_doc_id: str = None, original_category: str = None):
    SUPPORTED_EXTENSIONS = {".pdf", ".docx"}
    if not file_path.exists() or file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        print(f"[!] Bỏ qua (không hỗ trợ): {file_path.name}")
        return

    category = file_path.parent.name
    filename = file_path.name
    doc_id = file_path.stem
    parent_doc_id = original_doc_id if original_doc_id else doc_id  

    if original_doc_id and original_category:
        # File từ partition → lưu vào category/filename_partitioned_processed/
        base_folder = OUTPUT_DIR / original_category / f"{original_doc_id}_partitioned_processed"
    else:
        # File thường → lưu vào category/
        base_folder = OUTPUT_DIR / category

    year_match = re.search(r'\b(19|20)\d{2}\b', doc_id)
    year = year_match.group() if year_match else "N/A"

    print(f"\n[*] Đang xử lý: {filename} (Category: {category}, Year: {year})")

    temp_pdf = None
    upload_path = file_path
    if file_path.suffix.lower() == ".docx":
        print(f"    [~] Converting .docx → .pdf...")
        temp_pdf = convert_docx_to_temp_pdf(file_path)
        upload_path = temp_pdf  # upload file PDF thay vì docx

    try:
        file_upload = client.files.upload(file=upload_path)
        while file_upload.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(2)
            file_upload = client.files.get(name=file_upload.name)

        if file_upload.state.name == "FAILED":
            print(f"\n[X] Upload thất bại: {filename}")
            return

        formatted_instruction = SYSTEM_INSTRUCTION.format(
            filename=filename,
            doc_id=doc_id,
            parent_doc_id=parent_doc_id,   # ← thêm
            category=original_category or category,
            year=year
        )

        response = client.models.generate_content(
            model=MODEL,
            contents=[file_upload, "Hãy tiền xử lý tài liệu này theo đúng hướng dẫn."],
            config=types.GenerateContentConfig(
                system_instruction=formatted_instruction,
                temperature=0.1
            )
        )

        if "<<<CHUNK_START>>>" not in response.text:
            print(f"\n[!] Output không đúng format — lưu vào thư mục review")
            output_folder = OUTPUT_DIR / "_review" / (original_category or category)
        else:
            output_folder = base_folder

        output_folder.mkdir(parents=True, exist_ok=True)
        output_file = output_folder / f"{doc_id}_preprocessed.txt"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(response.text.strip())

        client.files.delete(name=file_upload.name)
        print(f"\n[V] Xong: {output_file}")

    except Exception as e:
        print(f"\n[X] Lỗi khi xử lý {filename}: {e}")

    finally:
        # Dọn file PDF tạm dù thành công hay lỗi
        if temp_pdf and temp_pdf.exists():
            temp_pdf.unlink()
            temp_pdf.parent.rmdir()

In [9]:
#functions to partition long files then dispatch them

def partition_and_process(file_path: Path):
    """Xử lý file lớn: partition nếu chưa có, rồi gửi từng part"""
    original_doc_id = file_path.stem
    original_category = file_path.parent.name
    partition_folder = file_path.parent / f"{original_doc_id}_partitioned"
    marker = partition_folder / "partitioned"

    if marker.exists():
        print("    [~] Tìm thấy partition cũ, dùng lại...")
        parts = sorted(f for f in partition_folder.iterdir()
                       if f.suffix.lower() in {".pdf", ".docx"})
    else:
        partition_folder.mkdir(parents=True, exist_ok=True)
        if file_path.suffix.lower() == ".pdf":
            structure = extract_structure_from_pdf(file_path)
            parts = split_pdf(file_path, structure, partition_folder)
        else:
            parts = split_docx(file_path, partition_folder)
        marker.touch()
        print(f"    [~] Đã tạo {len(parts)} parts tại {partition_folder.name}/")

    for part in parts:
        upload_and_process(part, original_doc_id=original_doc_id, original_category=original_category)


def check_and_dispatch(file_path: Path):
    """Entry point thay thế upload_and_process ở tầng trên"""
    file_path = Path(file_path)
    if not file_path.exists() or file_path.suffix.lower() not in {".pdf", ".docx"}:
        print(f"[!] Bỏ qua (không hỗ trợ): {file_path.name}")
        return

    page_count = get_page_count(file_path)
    suffix_note = "(ước tính)" if file_path.suffix.lower() == ".docx" else ""
    print(f"\n[*] {file_path.name} — {page_count} trang {suffix_note}")

    if page_count <= PAGE_THRESHOLD:
        upload_and_process(file_path)
    else:
        print(f"    [!] Vượt ngưỡng {PAGE_THRESHOLD} trang → partition mode")
        partition_and_process(file_path)

In [10]:
#main functions to process files

def process_single_file(path_str: str):
    check_and_dispatch(Path(path_str))   # ← đổi

def process_category(category_name: str):
    category_path = DATA_DIR / category_name
    if not category_path.is_dir():
        print(f"Lỗi: Category '{category_name}' không tồn tại")
        return
    files = list(category_path.glob("*.*"))
    print(f"Tìm thấy {len(files)} file trong nhóm {category_name}")
    for file in files:
        check_and_dispatch(file)         # ← đổi

def process_all():
    total = sum(1 for f in DATA_DIR.rglob("*.pdf"))
    print(f"Bắt đầu xử lý {total} file PDF trong toàn bộ thư mục Data...")
    for category_folder in DATA_DIR.iterdir():
        if category_folder.is_dir():
            process_category(category_folder.name)

In [11]:
#preprocess partitioned_file, dành cho trường hợp đang batch preprocess partitioned_file thì một vài file bị lỗi

def process_part_file(path_str: str):
    """Xử lý thủ công một file part trong folder _partitioned"""
    file_path = Path(path_str)
    parent = file_path.parent

    if parent.name.endswith("_partitioned"):
        original_doc_id = parent.name.replace("_partitioned", "")
        original_category = parent.parent.name
        upload_and_process(file_path,
                           original_doc_id=original_doc_id,
                           original_category=original_category)
    else:
        # Không phải part file, xử lý bình thường
        upload_and_process(file_path)

In [12]:
process_part_file(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2021_Chuan_dau_vao_partitioned\Ngoai_ngu_2021_Chuan_dau_vao_01.pdf")


[*] Đang xử lý: Ngoai_ngu_2021_Chuan_dau_vao_01.pdf (Category: Ngoai_ngu_2021_Chuan_dau_vao_partitioned, Year: N/A)

[V] Xong: ..\Preprocessed_Data\Ngoai_ngu\Ngoai_ngu_2021_Chuan_dau_vao_partitioned_processed\Ngoai_ngu_2021_Chuan_dau_vao_01_preprocessed.txt


In [24]:
#test
process_category("Huong_dan")

Tìm thấy 7 file trong nhóm Huong_dan

[*] Huong_dan_2020_chuyen_truong.docx — 1 trang (ước tính)

[*] Đang xử lý: Huong_dan_2020_chuyen_truong.docx (Category: Huong_dan, Year: N/A)
    [~] Converting .docx → .pdf...


100%|██████████| 1/1 [00:07<00:00,  7.14s/it]



[X] Lỗi khi xử lý Huong_dan_2020_chuyen_truong.docx: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

[*] Huong_dan_2021_Gui_cau_hoi.docx — 7 trang (ước tính)

[*] Đang xử lý: Huong_dan_2021_Gui_cau_hoi.docx (Category: Huong_dan, Year: N/A)
    [~] Converting .docx → .pdf...


100%|██████████| 1/1 [00:11<00:00, 11.51s/it]



[X] Lỗi khi xử lý Huong_dan_2021_Gui_cau_hoi.docx: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

[*] Huong_dan_2021_nhan_bang.docx — 1 trang (ước tính)

[*] Đang xử lý: Huong_dan_2021_nhan_bang.docx (Category: Huong_dan, Year: N/A)
    [~] Converting .docx → .pdf...


100%|██████████| 1/1 [00:05<00:00,  5.45s/it]



[X] Lỗi khi xử lý Huong_dan_2021_nhan_bang.docx: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

[*] Huong_dan_2021_Quy_doi_tin_chi.pdf — 7 trang 

[*] Đang xử lý: Huong_dan_2021_Quy_doi_tin_chi.pdf (Category: Huong_dan, Year: N/A)

[V] Xong: ..\Preprocessed_Data\Huong_dan\Huong_dan_2021_Quy_doi_tin_chi_preprocessed.txt

[*] Huong_dan_2023_Quy_doi_tin_chi.pdf — 4 trang 

[*] Đang xử lý: Huong_dan_2023_Quy_doi_tin_chi.pdf (Category: Huong_dan, Year: N/A)

[X] Lỗi khi xử lý Huong_dan_2023_Quy_doi_tin_chi.pdf: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

[*] Huong_dan_2023_Thuc_tap.pdf — 4 trang 

[*] Đang xử lý: Huong_dan_2023_Thuc_tap.pdf (Category: Huong_dan, Year: N/A)

[X] Lỗi khi xử lý Hu

In [16]:
process_single_file(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Do_an\DATN_2021_Dang_ky.docx")


[*] DATN_2021_Dang_ky.docx — 3 trang (ước tính)

[*] Đang xử lý: DATN_2021_Dang_ky.docx (Category: Do_an, Year: N/A)
    [~] Converting .docx → .pdf...


100%|██████████| 1/1 [00:04<00:00,  4.71s/it]



[V] Xong: ..\Preprocessed_Data\Do_an\DATN_2021_Dang_ky_preprocessed.txt
